# Exercise 2: From a test plan to full branch coverage

**Goal:** After this exercise, you can derive tests from a rule and reach full branch coverage.

**Scenario:** The map from exercise 1 shows that `rules.py` holds the penalty rule, and it has one test. You write a test plan from the rule. Codex turns the plan into tests, and coverage shows what the tests ran.

**Coding agent:** yes. You use Codex in a terminal.

**Where you work:** the folder `week-02/day-02/exercise-02-test-plan-to-branch-coverage/lab` in the course folder. You run the cells of this notebook and some commands in a terminal. You do not commit.

**Done when:** cell 3 prints `Done:`.

**Optional part:** add the rows of 3-value boundary analysis at the cap, 9 and 12, to the plan and to the tests.

**What you have:** in `lab`, the job as exercise 1 ends. `payment_job/rules.py` holds the rule, `tests/test_rules.py` holds one test, and `TEST_PLAN.md` holds a plan with one row.

**Start from:** step 1. To start again, run the reset command of step 1 again.

### Step 1. See the start

Open a terminal (**Terminal > New Terminal**). It opens in the course folder. Run:

```sh
source .venv/bin/activate
python3 reset.py exercise-02-test-plan-to-branch-coverage
```

Then run cell 1. It shows the plan's rows and runs the tests. You should see one row and `1 passed`.

In [ ]:
# 1. See the start. You do not need to read this code.
import subprocess
import sys
from pathlib import Path

lab = Path(_dh[0]) / "lab"
if not (lab / ".git").is_dir():
    print("lab is not there yet. Run the two commands of step 1 in a terminal.")
else:
    print("\n".join(line for line in (lab / "TEST_PLAN.md").read_text().splitlines() if line.startswith("|")))
    tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "--color=no"],
                           cwd=lab, capture_output=True, text=True)
    print(tests.stdout.strip().splitlines()[-1])

### Step 2. Write the test plan

In the Explorer, open `lab/TEST_PLAN.md`. Add one table row for each partition and boundary below. Work out each expected value by hand, from the rule at the top of the file. The rule: 1 percent of the unpaid tax for each month, for at most 10 months.

| Row | Why |
|---|---|
| `-1` months: `error` | "test cases must exercise all identified partitions (including invalid partitions)". |
| `0` months: `0` | On time is its own partition: no penalty. |
| `1` month and `10` months | The first and the last month of the partition 1 to 10. |
| `11` months: the same as `10` | The first capped month. "Two-value analysis tests each boundary and its neighbour across the edge." |
| `0` unpaid tax: `0` | Nothing owed is the second partition of `unpaid_tax`. |
| One account through `penalty_for`: any months and tax, in the same columns | `__main__.py` calls `penalty_for` for each account, and coverage measures all of `rules.py`. |

Save the file.

<details><summary>Stuck? One way to write the rows</summary>

```markdown
| invalid: negative months | -1 | 1000 | error |
| on time | 0 | 1000 | 0 |
| first month late | 1 | 1000 | 10 |
| inside 1 to 10 | 5 | 1000 | 50 |
| last month before the cap | 10 | 1000 | 100 |
| first capped month | 11 | 1000 | 100 |
| nothing owed | 5 | 0 | 0 |
| one account, through penalty_for | 4 | 2000 | 80 |
```

</details>

### Step 3. Have Codex turn the plan into tests

In the terminal, run:

```sh
cd week-02/day-02/exercise-02-test-plan-to-branch-coverage/lab
codex
```

If Codex asks if you trust this folder, answer yes. Write your request in your own words. Codex can replace the one test of the start: its row is in the plan. It must say these things:

| What your request says | Why |
|---|---|
| Turn `TEST_PLAN.md` into tests in `tests/test_rules.py`, with `@pytest.mark.parametrize` | "pytest runs the test once for each tuple." One test function holds the rows. |
| The error row gets its own test, with `pytest.raises` | "The negative row gets its own test, which expects the error." `pytest.raises` passes only when the code raises the error. |
| The `penalty_for` row gets its own test, with one `Account` | `penalty_for` takes an account, not two numbers. |
| Take each expected value from the plan, not from the code. Do not change `payment_job` | "An expected value taken from the code's output only proves the code agrees with itself." |

<details><summary>Stuck? One way to say it</summary>

`Turn TEST_PLAN.md into pytest tests in tests/test_rules.py. Use one test with @pytest.mark.parametrize for the rows of late_payment_penalty, and give the error row its own test with pytest.raises. Give the penalty_for row its own test, with one Account. Take each expected value from the plan, not from the code. Do not change payment_job.`

</details>

When Codex reports its tests, type `/quit`. Then press Enter. A test that fails shows a row whose plan and code do not agree. Check that row by hand against the rule. If the plan is wrong, correct the row in `TEST_PLAN.md` and in the test.

### Step 4. Measure branch coverage

In cell 2, set `command` to this command. Then run the cell.

| Type | Why |
|---|---|
| `command = "coverage run --branch -m pytest"` | Branch coverage counts the paths out of each decision. "To measure branch coverage, run coverage.py with the `--branch` flag". |

The cell then prints `coverage report`. `Branch` counts the branches. `BrPart` counts partial branches: a decision that the tests took one way only. `Missing` lists the lines that no test ran, and each jump no test took, such as `21->23`. You should see `100%` for `payment_job/rules.py`. If you do not, add the plan row that runs each missing line. Then do step 3 again.

In [ ]:
# 2. Measure branch coverage. You do not need to read the code below the first line.
command = ""

import subprocess
import sys
from pathlib import Path

lab = Path(_dh[0]) / "lab"
if not command.startswith("coverage run"):
    print('Type the command first: command = "coverage run --branch -m pytest". See step 4.')
else:
    run = command.split()[1:] + ["-q", "-p", "no:cacheprovider", "--color=no"]
    tests = subprocess.run([sys.executable, "-m", "coverage", *run], cwd=lab, capture_output=True, text=True)
    print(tests.stdout.strip().splitlines()[-1])
    print(subprocess.run([sys.executable, "-m", "coverage", "report"], cwd=lab, capture_output=True, text=True).stdout)

### Step 5. Check

Run cell 3. It prints `Done:` when these things are true:

- The plan has a row for each partition and boundary of step 2, and each expected value follows the rule.
- Each row of `late_payment_penalty` is in the parametrize list, the error row is in a `pytest.raises` test, and the tests pass.
- Branch coverage of `rules.py` is full.
- The tests fail on a copy of the code whose cap is 9 months.

Otherwise it prints `Not yet:` and each thing still to do. You do not need to read its code.

In [ ]:
# 3. Check. You do not need to read this code.
%run "{_dh[0]}/check.py"

### Afterwards, together

1. "The `min` line always goes to the same next line, so coverage.py sees no branch there." Which row of your plan did coverage not ask for, and why keep it?
2. Where did each expected value come from?
3. "Use partitions to decide which tests to write. Use branch coverage to see what those tests missed." Which rule of your own code has a cap that no test sits on?

<details><summary>Answer</summary>

The plan: one row for each partition, invalid ones too, and two rows at each edge: 0 and 1 months, 10 and 11 months. You work out each expected value by hand. 1 percent of 1,000 dollars is 10 dollars a month, and 11 months pays the same as 10.

The rows that coverage did not ask for: 10 and 11. The cap is one call to `min`, so coverage sees no branch there. Only these rows make the tests fail on a cap of 9 months.

The rows that coverage asked for: `-1`, which runs the error, `0` unpaid tax, which runs `return 0.0`, and the `penalty_for` row.

In the kept run, the plan's `penalty_for` row first said 40, and its test failed. The code was right: 4 months at 1 percent of 2,000 dollars is 80. A failed test can show a wrong plan.

The plan and the tests are in [`answers/exercise-02-test-plan-to-branch-coverage`](../answers/exercise-02-test-plan-to-branch-coverage/). Each reason, and where each quoted line comes from, is in [`decisions.md`](../answers/exercise-02-test-plan-to-branch-coverage/decisions.md).

</details>